# 001 · HTTP Requests with urllib

> **Standard Library › Networking** · Python 3.14

`urllib.request` is the standard library's HTTP client and `urllib.parse` its URL toolkit. They cover the same ground as `HttpClient` and `Uri`/`UriBuilder`/`HttpUtility` in .NET: send GET/POST requests, read status, headers and bodies, build query strings, and handle errors.

**In this notebook you will learn:**

- making GET requests with `urlopen` and reading status, headers and JSON bodies
- sending POST requests with JSON using `Request` (method, headers, data)
- handling `HTTPError` and `URLError`, and setting timeouts
- building and parsing URLs with `urllib.parse` (`urlencode`, `quote`, `urlparse`, `urljoin`, `parse_qs`)
- why most projects use the third-party `requests` or `httpx` libraries instead

## .NET mental model

| .NET | Python |
|---|---|
| `HttpClient.GetAsync(url)` | `urllib.request.urlopen(url)` (synchronous) |
| `HttpRequestMessage` | `urllib.request.Request` |
| `HttpResponseMessage.StatusCode` | `response.status` |
| `response.Content.ReadAsStringAsync()` | `response.read().decode()` |
| `response.Content.ReadFromJsonAsync<T>()` | `json.load(response)` |
| `EnsureSuccessStatusCode()` → `HttpRequestException` | raised automatically: `urllib.error.HTTPError` |
| `HttpClient.Timeout` | `urlopen(..., timeout=seconds)` |
| `Uri`, `UriBuilder` | `urllib.parse.urlparse` / `urlunparse` |
| `HttpUtility.ParseQueryString` / `QueryHelpers` | `urllib.parse.parse_qs` / `urlencode` |
| `Uri.EscapeDataString` | `urllib.parse.quote` |
| `new Uri(baseUri, relative)` | `urllib.parse.urljoin(base, relative)` |

`urllib` is **synchronous** and fairly low-level. It is perfect for scripts and for understanding HTTP, but in applications most Python developers use **`requests`** (simple, synchronous) or **`httpx`** (sync + async, HTTP/2) — both third-party packages installed with `uv add`. The concepts in this notebook carry over directly.

## A local test server

This notebook must work offline, so we start a tiny JSON API on `127.0.0.1` with port `0` (the OS picks a free port) in a **background daemon thread**. How the server works is the topic of *Standard Library › Networking › 002 · A Simple HTTP Server* — for now just run the cell.

In [ ]:
import json
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs

class DemoAPI(BaseHTTPRequestHandler):
    def send_json(self, status, payload):
        body = json.dumps(payload).encode()
        try:
            self.send_response(status)
            self.send_header("Content-Type", "application/json; charset=utf-8")
            self.send_header("Content-Length", str(len(body)))
            self.end_headers()
            self.wfile.write(body)
        except (BrokenPipeError, ConnectionResetError):
            pass                                   # client gave up (timeout demo)

    def do_GET(self):
        url = urlparse(self.path)
        match url.path:
            case "/api/products":
                self.send_json(200, [{"id": 1, "name": "keyboard", "price": 49.9},
                                     {"id": 2, "name": "mouse", "price": 19.5}])
            case "/api/echo":
                self.send_json(200, {"query": parse_qs(url.query),
                                     "accept": self.headers.get("Accept"),
                                     "agent": self.headers.get("User-Agent", "").split("/")[0]})
            case "/slow":
                time.sleep(0.3)
                self.send_json(200, {"slow": True})
            case "/boom":
                self.send_json(500, {"error": "something broke"})
            case _:
                self.send_json(404, {"error": f"no route for {url.path}"})

    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        data = json.loads(self.rfile.read(length))
        self.send_json(201, {"created": data, "content_type": self.headers.get_content_type()})

    def log_message(self, format, *args):          # keep the notebook output quiet
        pass

server = ThreadingHTTPServer(("127.0.0.1", 0), DemoAPI)
BASE = f"http://127.0.0.1:{server.server_address[1]}"
threading.Thread(target=server.serve_forever, daemon=True).start()
print("server running:", BASE.startswith("http://127.0.0.1:"))

## GET with `urlopen`

```csharp
// C#
using var http = new HttpClient();
using var response = await http.GetAsync($"{baseUrl}/api/products");
Console.WriteLine((int)response.StatusCode);
Console.WriteLine(response.Content.Headers.ContentType);
string text = await response.Content.ReadAsStringAsync();
```

`urlopen(url)` sends the request and returns an `http.client.HTTPResponse`. Use it as a context manager so the connection is closed (like `using`). The body is **bytes** — decode it yourself.

In [ ]:
from urllib.request import urlopen

with urlopen(f"{BASE}/api/products", timeout=5) as response:
    print(response.status, response.reason)              # 200 OK
    print(response.headers["Content-Type"])              # header lookup is case-insensitive
    raw = response.read()                                # bytes

print(type(raw).__name__, raw[:30])
products = json.loads(raw.decode("utf-8"))
print(products[0]["name"], products[1]["price"])

**Tip:** `json.load(response)` reads and parses in one step (it accepts a binary file-like object and detects UTF-8 automatically). And `response.headers.get_content_charset()` tells you the declared charset when you need to decode text.

In [ ]:
with urlopen(f"{BASE}/api/products", timeout=5) as response:
    print(response.headers.get_content_type(), response.headers.get_content_charset())
    products = json.load(response)

print([p["name"] for p in products])

⚠️ **Gotcha:** `read()` can only be called once — the body is a stream, like `ReadAsStreamAsync()`. Store the result if you need it twice.

## Query strings with `urlencode`

```csharp
// C#
var query = HttpUtility.ParseQueryString(string.Empty);
query["q"] = "café & bar";
query["page"] = "2";
var url = $"{baseUrl}/api/echo?{query}";
```

Never concatenate user input into a URL by hand. `urllib.parse.urlencode` takes a dict (or list of pairs) and percent-encodes it; `doseq=True` turns a list value into repeated keys.

In [ ]:
from urllib.parse import urlencode

params = {"q": "café & bar", "page": 2, "tag": ["new", "sale"]}
qs = urlencode(params, doseq=True)
print(qs)

with urlopen(f"{BASE}/api/echo?{qs}", timeout=5) as response:
    print(json.load(response)["query"])                  # the server decoded it back

## Custom requests: headers, POST and JSON

```csharp
// C#
using var request = new HttpRequestMessage(HttpMethod.Post, $"{baseUrl}/api/orders")
{
    Content = JsonContent.Create(new { Product = "keyboard", Qty = 2 })
};
request.Headers.Add("Accept", "application/json");
using var response = await http.SendAsync(request);
```

`urllib.request.Request(url, data=..., headers=..., method=...)` is the `HttpRequestMessage`. `data` must be **bytes**; when `data` is given, the default method is `POST`.

In [ ]:
from urllib.request import Request

req = Request(f"{BASE}/api/echo", headers={"Accept": "application/json"})
with urlopen(req, timeout=5) as response:
    echo = json.load(response)
print(echo["accept"], "| default User-Agent:", echo["agent"])   # Python-urllib

In [ ]:
order = {"product": "keyboard", "qty": 2}

req = Request(
    f"{BASE}/api/orders",
    data=json.dumps(order).encode("utf-8"),               # body as bytes
    headers={"Content-Type": "application/json", "Accept": "application/json"},
    method="POST",
)
with urlopen(req, timeout=5) as response:
    print(response.status)                                # 201 Created
    print(json.load(response))

⚠️ **Gotcha:** if you pass `data` without a `Content-Type` header, urllib sends `application/x-www-form-urlencoded`. For a JSON API, always set `Content-Type: application/json` explicitly. For an HTML-style form post, use `data=urlencode(fields).encode()`.

**Tip:** `method=` accepts any verb — `"PUT"`, `"PATCH"`, `"DELETE"`, `"HEAD"`.

## Errors: `HTTPError` and `URLError`

Unlike `HttpClient` (which returns a 404/500 response and only throws if you call `EnsureSuccessStatusCode()`), `urlopen` **raises** for every 4xx/5xx status:

- `urllib.error.HTTPError` — the server answered with an error status. It has `.code`, `.reason`, `.headers` and the error body via `.read()`.
- `urllib.error.URLError` — no HTTP response at all (DNS failure, connection refused, ...); `.reason` holds the underlying error.

`HTTPError` is a subclass of `URLError`, so catch it **first**.

In [ ]:
from urllib.error import HTTPError, URLError

for path in ["/missing", "/boom"]:
    try:
        urlopen(f"{BASE}{path}", timeout=5)
    except HTTPError as e:
        body = json.loads(e.read())
        print(f"{type(e).__name__}: {e.code} {e.reason} -> {body['error']}")
        e.close()

To trigger a `URLError` offline, we connect to a port where nothing is listening (we briefly grab a free port from the OS, then release it):

In [ ]:
import socket

with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    closed_port = s.getsockname()[1]          # free port, nothing listening after the with

try:
    urlopen(f"http://127.0.0.1:{closed_port}/", timeout=2)
except HTTPError as e:
    print("HTTP error", e.code)
except URLError as e:
    print(f"{type(e).__name__}: reason = {type(e.reason).__name__}")

## Timeouts

⚠️ **Gotcha:** without `timeout=`, `urlopen` uses the global socket default, which is **no timeout** — a hung server blocks your program forever. (`HttpClient` defaults to 100 seconds.) Always pass a timeout.

A timeout while waiting for the response raises the built-in `TimeoutError`; a timeout while connecting arrives wrapped in a `URLError`. Catch both:

In [ ]:
start = time.perf_counter()
try:
    urlopen(f"{BASE}/slow", timeout=0.1)       # the server needs 0.3s
except TimeoutError as e:
    print(f"{type(e).__name__}: {e}")
except URLError as e:
    print(f"URLError: {e.reason}")
print(f"gave up after ~{time.perf_counter() - start:.1f}s")

## Parsing and building URLs with `urllib.parse`

```csharp
// C#
var uri = new Uri("https://shop.example.com:8443/api/v1/items?q=1&q=2#top");
Console.WriteLine(uri.Host);     // shop.example.com
Console.WriteLine(uri.Port);     // 8443
var b = new UriBuilder(uri) { Query = "page=2" };
```

`urlparse` splits a URL into a named tuple (`scheme`, `netloc`, `path`, `params`, `query`, `fragment`) with convenience properties like `hostname` and `port`. Because it's a named tuple, `_replace()` + `geturl()` play the role of `UriBuilder`.

In [ ]:
from urllib.parse import urlparse, parse_qs, parse_qsl

u = urlparse("https://shop.example.com:8443/api/v1/items?q=1&q=2&sort=price#top")
print(u.scheme, u.hostname, u.port, u.path, u.fragment)
print(parse_qs(u.query))          # dict of lists (keys can repeat)
print(parse_qsl(u.query))         # list of pairs, order kept

new_url = u._replace(query=urlencode({"page": 2}), fragment="").geturl()
print(new_url)

### Encoding pieces: `quote` and `quote_plus`

- `quote(s)` percent-encodes for use in a **path** segment; `/` is kept by default (pass `safe=""` to encode it too) — like `Uri.EscapeDataString`.
- `quote_plus(s)` encodes for a **query string** (spaces become `+`) — like `WebUtility.UrlEncode`.
- `unquote` / `unquote_plus` reverse them.

In [ ]:
from urllib.parse import quote, quote_plus, unquote

name = "summer sale/2026"
print(quote(name))               # summer%20sale/2026
print(quote(name, safe=""))      # summer%20sale%2F2026  — safe for a single path segment
print(quote_plus(name))          # summer+sale%2F2026
print(unquote("caf%C3%A9"))      # café

### Resolving relative URLs with `urljoin`

`urljoin(base, relative)` follows browser rules, like `new Uri(baseUri, relative)` in .NET.

⚠️ **Gotcha:** the trailing slash on the base matters — without it, the last segment is *replaced*. And a relative path starting with `/` replaces the whole path.

In [ ]:
from urllib.parse import urljoin

print(urljoin("https://shop.example.com/api/v1/", "items"))   # .../api/v1/items
print(urljoin("https://shop.example.com/api/v1", "items"))    # .../api/items   (v1 replaced!)
print(urljoin("https://shop.example.com/api/v1/", "/health")) # .../health       (absolute path)
print(urljoin("https://shop.example.com/a/b/", "../c"))       # .../a/c

## Shutting down the test server

`shutdown()` stops the `serve_forever` loop (it must be called from another thread — here, the notebook's main thread) and `server_close()` releases the listening socket.

In [ ]:
server.shutdown()
server.server_close()
print("server stopped")

## A note on `requests` and `httpx`

For real projects, the third-party libraries are much more pleasant:

```python
# uv add requests          (not part of the standard library)
import requests

r = requests.get(f"{BASE}/api/products", params={"page": 2}, timeout=5)
r.raise_for_status()                       # like EnsureSuccessStatusCode()
products = r.json()

r = requests.post(f"{BASE}/api/orders", json={"product": "keyboard", "qty": 2}, timeout=5)
```

`httpx` has an almost identical API plus `httpx.AsyncClient` for use with `asyncio` — the closest thing to `HttpClient` with `async`/`await`. Reach for `urllib` when you can't add dependencies (small scripts, tooling, restricted environments).

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| GET | `await http.GetAsync(url)` | `with urlopen(url, timeout=5) as r:` |
| Status | `response.StatusCode` | `r.status` |
| Header | `response.Headers` / `Content.Headers` | `r.headers["Content-Type"]` |
| Body as text | `ReadAsStringAsync()` | `r.read().decode()` |
| Body as JSON | `ReadFromJsonAsync<T>()` | `json.load(r)` |
| Custom request | `HttpRequestMessage` | `Request(url, data=, headers=, method=)` |
| POST JSON | `JsonContent.Create(obj)` | `data=json.dumps(obj).encode()` + `Content-Type` |
| Error status | `HttpRequestException` (after `EnsureSuccess...`) | `HTTPError` (always raised) |
| Network failure | `HttpRequestException` | `URLError` |
| Timeout | `HttpClient.Timeout` | `timeout=` → `TimeoutError` |
| Build query | `HttpUtility.ParseQueryString` | `urlencode(dict, doseq=True)` |
| Parse query | `HttpUtility.ParseQueryString` | `parse_qs` / `parse_qsl` |
| Escape | `Uri.EscapeDataString` | `quote(s, safe="")` |
| Parse URL | `new Uri(s)` | `urlparse(s)` |
| Relative URL | `new Uri(base, rel)` | `urljoin(base, rel)` |

## Exercises

The solutions start their own small server, so they run on their own.

1. Write `get_json(url, timeout=5)` that returns the parsed JSON body, or `None` if the server answers with an HTTP error status. Test it against a route that returns JSON and one that returns 404.
2. Build the URL `https://api.example.com/search?q=red+shoes&size=42&size=43` from `"https://api.example.com"`, the path `"search"` and the dict `{"q": "red shoes", "size": [42, 43]}` using `urljoin` and `urlencode`.
3. Write `post_json(url, payload)` that sends `payload` as JSON with a `POST` and returns `(status, parsed_body)`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import json, threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.request import urlopen, Request
from urllib.error import HTTPError
from urllib.parse import urljoin, urlencode

# a minimal server for testing the solutions
class Api(BaseHTTPRequestHandler):
    def reply(self, status, payload):
        body = json.dumps(payload).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)
    def do_GET(self):
        if self.path == "/ok":
            self.reply(200, {"status": "ok"})
        else:
            self.reply(404, {"error": "not found"})
    def do_POST(self):
        data = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        self.reply(201, {"echo": data})
    def log_message(self, *args):
        pass

srv = ThreadingHTTPServer(("127.0.0.1", 0), Api)
threading.Thread(target=srv.serve_forever, daemon=True).start()
base = f"http://127.0.0.1:{srv.server_address[1]}"

# 1
def get_json(url, timeout=5):
    try:
        with urlopen(url, timeout=timeout) as r:
            return json.load(r)
    except HTTPError as e:
        e.close()
        return None

print(get_json(f"{base}/ok"), get_json(f"{base}/nope"))

# 2
url = urljoin("https://api.example.com", "search") + "?" + urlencode(
    {"q": "red shoes", "size": [42, 43]}, doseq=True)
print(url)

# 3
def post_json(url, payload):
    req = Request(url, data=json.dumps(payload).encode(),
                  headers={"Content-Type": "application/json"}, method="POST")
    with urlopen(req, timeout=5) as r:
        return r.status, json.load(r)

print(post_json(f"{base}/orders", {"sku": "A-1", "qty": 3}))

srv.shutdown()
srv.server_close()
```

</details>